In [1]:
import pandas as pd
import numpy as np

COLUMNS = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num",
]
df=pd.read_csv("../data/raw/processed.cleveland.data", header=None, names=COLUMNS, na_values="?")
print("dimension de dataset est:",df.shape)
print(df.head())

dimension de dataset est: (303, 14)
    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  num  
0    3.0  0.0   6.0    0  
1    2.0  3.0   3.0    2  
2    2.0  2.0   7.0    1  
3    3.0  0.0   3.0    0  
4    1.0  0.0   3.0    0  


In [2]:
print("Types des colonnes")
print(df.dtypes)
print("\n Nombre de valeurs manquantes par colonne")
missing = df.isna().sum()
print(missing[missing > 0])

Types des colonnes
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
dtype: object

 Nombre de valeurs manquantes par colonne
ca      4
thal    2
dtype: int64


In [3]:
# Vérifier s'il y a des doublons stricts dans le dataset
nb_doublons = df.duplicated().sum()
print(f"Nombre de lignes en double : {nb_doublons}")

Nombre de lignes en double : 0


In [4]:
# 1. Vérification de sécurité : vérifier que 'num' ne contient que 0, 1, 2, 3 ou 4
if not df["num"].isin([0, 1, 2, 3, 4]).all():
    raise ValueError("Valeur de cible inattendue.")

# 2. Binarisation : si num > 0 -> 1, sinon -> 0
y = (df["num"] > 0).astype(int)

# 3. Isoler les 13 variables explicatives dans X (on retire la cible 'num')
X = df.drop(columns="num")

# 4. Analyser la répartition des deux classes
print("--- Répartition en effectifs ---")
print(y.value_counts().rename({0: "Classe 0 (Absence)", 1: "Classe 1 (Présence)"}))

print("\n--- Répartition en pourcentages ---")
print(y.value_counts(normalize=True).rename({0: "Classe 0 (Absence)", 1: "Classe 1 (Présence)"}) * 100)

--- Répartition en effectifs ---
num
Classe 0 (Absence)     164
Classe 1 (Présence)    139
Name: count, dtype: int64

--- Répartition en pourcentages ---
num
Classe 0 (Absence)     54.125413
Classe 1 (Présence)    45.874587
Name: proportion, dtype: float64


## Séparation des données d’entraînement et de test (tâche 04)

Nous réservons **20 % des observations au test** et utilisons les **80 % restants pour l’entraînement**.

- `X` et `y` sont séparés ensemble afin de garder chaque observation associée à sa cible.
- `stratify=y` conserve approximativement la proportion des classes 0 et 1 dans chaque ensemble.
- `random_state=42` permet de reproduire la séparation pour les mêmes données dans le même ordre.
- Les valeurs manquantes sont conservées : l’imputation et l’encodage seront ajustés uniquement sur l’entraînement dans le futur pipeline. Le test est réservé à l’évaluation finale et ne sert pas à choisir le modèle.

Exécuter les cellules précédentes pour créer `X` et `y` avant cette étape.


In [5]:
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20
RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Entraînement : {X_train.shape[0]} observations, {X_train.shape[1]} variables")
print(f"Test : {X_test.shape[0]} observations, {X_test.shape[1]} variables")

repartition = pd.DataFrame({
    "Total (effectif)": y.value_counts(),
    "Entraînement (effectif)": y_train.value_counts(),
    "Test (effectif)": y_test.value_counts(),
    "Total (%)": y.value_counts(normalize=True) * 100,
    "Entraînement (%)": y_train.value_counts(normalize=True) * 100,
    "Test (%)": y_test.value_counts(normalize=True) * 100,
}).sort_index()
repartition.index.name = "Classe"
print("\nRépartition des classes (0 = absence, 1 = présence) :")
print(repartition.round(2).to_string())


Entraînement : 242 observations, 13 variables
Test : 61 observations, 13 variables

Répartition des classes (0 = absence, 1 = présence) :
        Total (effectif)  Entraînement (effectif)  Test (effectif)  Total (%)  Entraînement (%)  Test (%)
Classe                                                                                                   
0                    164                      131               33      54.13             54.13      54.1
1                    139                      111               28      45.87             45.87      45.9


In [6]:
# Vérifier la correspondance entre les mesures et les cibles.
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)

# Aucune observation ne doit appartenir aux deux ensembles.
assert X_train.index.intersection(X_test.index).empty
assert len(X_train) + len(X_test) == len(X)
assert X_train.index.union(X_test.index).equals(X.index.sort_values())
assert X_train.columns.equals(X.columns)
assert X_test.columns.equals(X.columns)
assert set(y_train.unique()) == set(y_test.unique()) == {0, 1}

print("Vérifications réussies : ensembles disjoints, complets et cibles alignées.")


Vérifications réussies : ensembles disjoints, complets et cibles alignées.
